# Explore GT export

Quick sanity checks on a faceiq-labs export. Run the export first (see README), then execute cells top to bottom.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from faceiq_pref.data import load_export

EXPORT_DIR = ROOT / 'data' / 'exports' / 'cmr1mr0m7000196d57zi3vcgn'
export = load_export(EXPORT_DIR)
export.manifest['counts']

In [ ]:
import pandas as pd

matchups = export.all_matchups()
df = pd.DataFrame([m.__dict__ for m in matchups])
print(len(df), 'matchups')
df['final_outcome'].value_counts()

In [ ]:
# Confidence distribution and human-audit overlap
print(df['confidence'].value_counts(dropna=False))
print()
print('human-labeled rows:', df['human_labeled_at'].notna().sum())
print('human overrides:', df['is_human_override'].sum())

In [ ]:
# Image coverage + a peek at a few faces
from PIL import Image
import matplotlib.pyplot as plt

faces = export.faces()
present, missing = export.image_coverage(faces)
print(f'{present}/{len(faces)} images present, {len(missing)} missing')

sample = [f for f in list(faces.values())[:6] if export.image_path(f).exists()]
fig, axes = plt.subplots(1, len(sample), figsize=(3 * len(sample), 3))
for ax, face in zip(axes, sample):
    ax.imshow(Image.open(export.image_path(face)))
    ax.set_title(f'{face.gender} D{face.decile_bin}', fontsize=9)
    ax.axis('off')